# 📡 Project 9.3 — Sensor Network Analysis
**DSA for Mechatronics · Week 09 — Graphs**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib, heapq
from collections import defaultdict, deque, Counter
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A wireless sensor network has nodes that communicate in pairs.
We need to:
1. Find **connected components** (isolated sensor clusters)
2. Check if the network is **bipartite** — can sensors be split into two groups
   where every link connects a group-A sensor to a group-B sensor?
   (This matters for frequency-division duplex: group A transmits on f1, group B on f2.)
3. Detect **bridges** (links whose removal disconnects the network) — critical links
4. Count **articulation points** (nodes whose removal disconnects the network)


## Step 1 — Build the sensor network

In [ ]:
# ── Personalised parameters ──────────────────────────────────────
N_NODES  = random.randint(12, 20)
N_EXTRA  = random.randint(3, 8)

NODE_NAMES = [f"N{i+1:02d}" for i in range(N_NODES)]

# Build random graph (not necessarily connected — intentional)
all_edges = set()
# First, build 2-3 components
n_comps   = random.randint(2, 3)
comp_sizes = []
remaining  = N_NODES
for ci in range(n_comps):
    size = random.randint(3, max(3, remaining - (n_comps - ci - 1)*3))
    comp_sizes.append(size)
    remaining -= size
    if remaining <= 0: break

# assign nodes to components
nodes_shuffled = list(range(N_NODES))
random.shuffle(nodes_shuffled)
components_init = []
idx = 0
for size in comp_sizes:
    components_init.append(nodes_shuffled[idx:idx+size])
    idx += size
if idx < N_NODES:
    components_init[-1].extend(nodes_shuffled[idx:])

# build edges within each component
for comp in components_init:
    comp_list = list(comp)
    random.shuffle(comp_list)
    for i in range(1, len(comp_list)):
        u, v = comp_list[i-1], comp_list[i]
        all_edges.add((min(u,v), max(u,v)))
    for _ in range(N_EXTRA // n_comps):
        u, v = random.sample(comp_list, 2)
        all_edges.add((min(u,v), max(u,v)))

all_edges = list(all_edges)

# Adjacency list
net = defaultdict(list)
for u, v in all_edges:
    net[u].append(v)
    net[v].append(u)
for i in range(N_NODES):
    net[i].sort()

print(f"Sensor network:")
print(f"  Nodes (sensors)  : {N_NODES}")
print(f"  Links (edges)    : {len(all_edges)}")
print()
print(f"  Adjacency list:")
for i in range(N_NODES):
    nbrs = [NODE_NAMES[j] for j in net[i]]
    deg  = len(nbrs)
    print(f"    {NODE_NAMES[i]} (deg={deg}): {nbrs}")


## Step 2 — Connected components + bipartite check

In [ ]:
def find_components_and_bipartite(net, n):
    """
    Single BFS pass:
    - Finds connected components.
    - Checks bipartiteness of each component using 2-coloring.
      Assign color 0 to start, color 1 to neighbours.
      If a neighbour has the same color → not bipartite.
    Returns: (components, is_bipartite, coloring)
    """
    color      = [-1] * n   # -1 = unvisited
    components = []
    is_bip     = True

    for start in range(n):
        if color[start] != -1:
            continue
        # BFS from this start node
        component = []
        queue     = deque([start])
        color[start] = 0
        while queue:
            node = queue.popleft()
            component.append(node)
            for nbr in net[node]:
                if color[nbr] == -1:
                    color[nbr] = 1 - color[node]
                    queue.append(nbr)
                elif color[nbr] == color[node]:
                    is_bip = False   # same color neighbours = not bipartite
        components.append(sorted(component))

    return components, is_bip, color

components, is_bip, coloring = find_components_and_bipartite(net, N_NODES)

print(f"Connected components: {len(components)}")
for ci, comp in enumerate(components, 1):
    names = [NODE_NAMES[j] for j in comp]
    print(f"  Component {ci} ({len(comp)} nodes): {names}")

print()
print(f"Bipartite check: {'✅ YES — valid frequency split' if is_bip else '❌ NO — odd-length cycle detected'}")
if is_bip:
    group_a = [NODE_NAMES[i] for i in range(N_NODES) if coloring[i] == 0]
    group_b = [NODE_NAMES[i] for i in range(N_NODES) if coloring[i] == 1]
    print(f"  Group A (tx on f1): {group_a}")
    print(f"  Group B (tx on f2): {group_b}")
else:
    print(f"  Cannot split sensors into two non-interfering frequency groups.")

# Degree distribution
degrees = [len(net[i]) for i in range(N_NODES)]
print(f"\nDegree distribution:")
deg_dist = Counter(degrees)
for deg in sorted(deg_dist):
    bar = "█" * deg_dist[deg]
    print(f"  deg {deg}: {deg_dist[deg]} nodes  {bar}")
print(f"  Max degree: {max(degrees)}  (node {NODE_NAMES[degrees.index(max(degrees))]})")
print(f"  Avg degree: {sum(degrees)/N_NODES:.2f}")


## Step 3 — Bridge detection (Tarjan's algorithm)

In [ ]:
def find_bridges(net, n):
    """
    Find bridges using Tarjan's bridge-finding algorithm.

    Each node gets:
      disc[u]  = discovery time (DFS timestamp)
      low[u]   = lowest disc reachable via DFS subtree + back edges

    An edge (u, v) is a bridge if low[v] > disc[u]:
    i.e., v cannot reach u or any ancestor of u without using edge (u,v).

    O(V + E) time.
    """
    disc   = [-1] * n
    low    = [-1] * n
    timer  = [0]
    bridges = []
    visited = [False] * n

    def dfs(u, parent):
        visited[u]  = True
        disc[u] = low[u] = timer[0]
        timer[0] += 1
        for v in net[u]:
            if not visited[v]:
                dfs(v, u)
                low[u] = min(low[u], low[v])
                if low[v] > disc[u]:
                    bridges.append((u, v))
            elif v != parent:
                low[u] = min(low[u], disc[v])

    for start in range(n):
        if not visited[start]:
            dfs(start, -1)
    return bridges

bridges = find_bridges(net, N_NODES)

print(f"Bridge detection (critical links):")
print(f"  Total bridges found: {len(bridges)}")
if bridges:
    print(f"  Removing any of these links will disconnect the network:")
    print(f"  {'Link':<20}  Consequence")
    print(f"  {'─'*20}  {'─'*40}")
    for u, v in bridges:
        print(f"  {NODE_NAMES[u]} ↔ {NODE_NAMES[v]:<15}  ⚠ Network splits if removed")
else:
    print(f"  ✅ No bridges — every link has a redundant alternative path.")


## Step 4 — Articulation points (Tarjan's)

In [ ]:
def find_articulation_points(net, n):
    """
    Find articulation points via DFS.
    Node u is an AP if:
      - u is the DFS root and has ≥ 2 children, OR
      - u is not root and has a child v with low[v] >= disc[u]
        (v cannot reach above u without going through u).
    """
    disc     = [-1] * n
    low      = [-1] * n
    parent   = [-1] * n
    is_ap    = [False] * n
    timer    = [0]
    visited  = [False] * n

    def dfs(u):
        visited[u]   = True
        disc[u] = low[u] = timer[0]
        timer[0] += 1
        child_count  = 0
        for v in net[u]:
            if not visited[v]:
                child_count += 1
                parent[v]    = u
                dfs(v)
                low[u] = min(low[u], low[v])
                if parent[u] == -1 and child_count > 1:
                    is_ap[u] = True
                if parent[u] != -1 and low[v] >= disc[u]:
                    is_ap[u] = True
            elif v != parent[u]:
                low[u] = min(low[u], disc[v])

    for start in range(n):
        if not visited[start]:
            dfs(start)
    return [i for i in range(n) if is_ap[i]]

aps = find_articulation_points(net, N_NODES)

print(f"Articulation points (critical nodes):")
print(f"  Total found: {len(aps)}")
if aps:
    print(f"  Removing any of these nodes will disconnect the network:")
    for ap in aps:
        nbr_count = len(net[ap])
        print(f"    {NODE_NAMES[ap]}  (degree {nbr_count})")
else:
    print(f"  ✅ No articulation points — network is 2-connected (biconnected).")

print()
# Summary counts
print(f"Network resilience summary:")
print(f"  Components           : {len(components)}")
print(f"  Bridges              : {len(bridges)}")
print(f"  Articulation points  : {len(aps)}")
redundant_nodes = [i for i in range(N_NODES) if i not in aps]
print(f"  Redundant nodes      : {len(redundant_nodes)}")
bip_str = "YES" if is_bip else "NO"
print(f"  Bipartite            : {bip_str}")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " SENSOR NETWORK ANALYSIS — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  NETWORK STRUCTURE" + " "*(W-19) + "║")
print(f"║  {'Sensors (nodes)':<28}: {N_NODES:<26}║")
print(f"║  {'Links (edges)':<28}: {len(all_edges):<26}║")
print(f"║  {'Connected components':<28}: {len(components):<26}║")
print(f"║  {'Bipartite (freq split)':<28}: {'YES ✅' if is_bip else 'NO ❌':<26}║")
print("╠" + "═"*W + "╣")
print("║  RESILIENCE" + " "*(W-12) + "║")
print(f"║  {'Bridges (critical links)':<28}: {len(bridges):<26}║")
print(f"║  {'Articulation points':<28}: {len(aps):<26}║")
print(f"║  {'Max node degree':<28}: {max(degrees):<26}║")
print(f"║  {'Avg node degree':<28}: {sum(degrees)/N_NODES:.2f}{'':<22}║")
if aps:
    ap_names = ", ".join(NODE_NAMES[i] for i in aps[:4])
    print(f"║  {'Critical nodes':<28}: {ap_names[:26]:<26}║")
if bridges:
    br_str = ", ".join(f"{NODE_NAMES[u]}↔{NODE_NAMES[v]}" for u,v in bridges[:3])
    print(f"║  {'Critical links':<28}: {br_str[:26]:<26}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about the network or system?

*Your answer here:*

---

**Q2 — Which graph concept did you find most important, and why?**
Pick one algorithm (BFS, DFS, topological sort, Dijkstra, cycle detection…) and explain what problem it solves.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
